In [2]:
# ==============================================================================
# 2026改修版: PLS (設計基準 A〜K 完全準拠 / SVM_Radial構成互換)
# - 重要度: PLS VIP (Variable Importance in Projection)
# ==============================================================================

suppressPackageStartupMessages({
  library(caret)
  library(dplyr)
  library(Metrics)
  library(pls)
})

set.seed(42)

# -------------------------------
# (F)(G) 設定：パスと対象ファイル
# -------------------------------
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
file_names <- c(
  "n_base.csv", "n_base_OH_rebuilt.csv", "n_base_FP_rebuilt.csv",
  "n_all.csv",  "n_all_OH_rebuilt.csv",  "n_all_FP_rebuilt.csv",
  "m_base.csv", "m_base_OH_rebuilt.csv", "m_base_FP_rebuilt.csv",
  "m_all.csv",  "m_all_OH_rebuilt.csv",  "m_all_FP_rebuilt.csv"
)

# 出力先設定 (F)
today <- format(Sys.Date(), "%Y%m%d")
out_root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results"
model_name <- "PLS"
run_dir <- file.path(out_root, paste0("Corr_1000_", model_name))
if (!dir.exists(run_dir)) dir.create(run_dir, recursive = TRUE)

# (C)(D)(E) 除外変数設定
inappropriate_vars <- c(
  'Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
  'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
  'IonIoffF', 'DRMS', 'Var.1'
)
physical_exclude_patterns <- c("X2DGIWAXD", "X2DGIXD", "widthfibril")

# -------------------------------
# HELPERS
# -------------------------------
safe_r2 <- function(y, p) {
  r <- suppressWarnings(cor(y, p))
  if (is.na(r)) return(NA_real_)
  return(as.numeric(r^2))
}

# (J) PLS VIP 計算関数
calc_pls_vip <- function(model_caret) {
  pls_mvr <- model_caret$finalModel
  ncomp <- model_caret$bestTune$ncomp
  
  W <- as.matrix(pls_mvr$loading.weights[, 1:ncomp, drop = FALSE])
  Tt <- as.matrix(pls_mvr$scores[, 1:ncomp, drop = FALSE])
  Q  <- as.matrix(pls_mvr$Yloadings[, 1:ncomp, drop = FALSE])
  
  p <- nrow(W)
  A <- ncol(W)
  SS <- numeric(A)
  for (a in 1:A) {
    SS[a] <- sum(Tt[, a]^2) * sum(Q[, a]^2)
  }
  
  SSY <- sum(SS)
  Wnorm2 <- colSums(W^2)
  vip <- numeric(p)
  for (j in 1:p) {
    wsum <- 0
    for (a in 1:A) {
      wsum <- wsum + SS[a] * (W[j, a]^2 / Wnorm2[a])
    }
    vip[j] <- sqrt(p * wsum / SSY)
  }
  data.frame(Feature = rownames(W), Importance = as.numeric(vip))
}

# -------------------------------
# MAIN LOOP
# -------------------------------
summary_list <- list()
importance_list <- list()

for (file in file_names) {
  filepath <- file.path(base_path, file)
  if (!file.exists(filepath)) next
  
  df_raw <- tryCatch(read.csv(filepath, row.names = 1, check.names = FALSE), error = function(e) NULL)
  if (is.null(df_raw)) next
  if ("X" %in% colnames(df_raw)) df_raw$X <- NULL

  for (target in target_vars) {
    if (!target %in% colnames(df_raw)) next

    # クレンジング
    df_temp <- df_raw[, sapply(df_raw, is.numeric)] %>% na.omit()
    if (nrow(df_temp) < 20) next

    # (C)(D)(E) 変数除外
    features <- setdiff(colnames(df_temp), target_vars)
    features <- setdiff(features, inappropriate_vars)
    features <- features[!grepl(paste(physical_exclude_patterns, collapse="|"), features)]
    
    # ゼロ分散除外
    vars <- sapply(df_temp[, features, drop = FALSE], var)
    features <- names(vars)[vars > 0 & !is.na(vars)]

    # (H) 多重共線性対策 (Corr > 0.99999)
    if (length(features) > 1) {
      cor_mat <- cor(df_temp[, features, drop = FALSE], use = "pairwise.complete.obs")
      cor_mat[is.na(cor_mat)] <- 0
      high_corr <- findCorrelation(cor_mat, cutoff = 0.99999)
      if (length(high_corr) > 0) features <- features[-high_corr]
    }
    if (length(features) == 0) next

    # (I) スケーリング [-1, 1]
    pp <- preProcess(df_temp[, features, drop = FALSE], method = "range")
    df_scaled <- df_temp
    df_scaled[, features] <- predict(pp, df_temp[, features]) * 2 - 1

    # (K) CV10設定
    train_ctrl <- trainControl(method = "cv", number = 10, savePredictions = "final")
    max_ncomp <- min(15, nrow(df_scaled) - 1, length(features))
    tune_grid <- expand.grid(ncomp = 1:max_ncomp)

    # 学習
    model <- tryCatch(
      train(
        x = df_scaled[, features, drop = FALSE], y = df_scaled[[target]],
        method = "pls", trControl = train_ctrl, tuneGrid = tune_grid, metric = "RMSE"
      ),
      error = function(e) { cat("  [ERROR] PLS failed:", e$message, "\n"); return(NULL) }
    )
    if (is.null(model)) next

    # --- 保存処理 ---
    tag <- paste0(tools::file_path_sans_ext(file), "_", target)
    
    # 1. モデルRDS
    saveRDS(model, file.path(run_dir, paste0(tag, "_model.rds")))

    # 2. (B) CV10_OOF予測CSV
    pred_oof <- model$pred %>% 
      filter(ncomp == model$bestTune$ncomp) %>%
      mutate(SampleID = rownames(df_scaled)[rowIndex], Type = "CV10_OOF") %>%
      select(SampleID, Observed = obs, Predicted = pred, Type)
    write.csv(pred_oof, file.path(run_dir, paste0(tag, "_pred_OOF.csv")), row.names = FALSE)

    # 3. (J) 重要度 (VIP)
    imp_df <- tryCatch(calc_pls_vip(model), error = function(e) NULL)
    if (!is.null(imp_df)) {
      imp_df$File <- file
      imp_df$Target <- target
      importance_list[[length(importance_list)+1]] <- imp_df
    }

    # 4. サマリー集計
    summary_list[[length(summary_list)+1]] <- data.frame(
      File = file, Target = target, 
      CV10_R2 = safe_r2(pred_oof$Observed, pred_oof$Predicted),
      CV10_RMSE = rmse(pred_oof$Observed, pred_oof$Predicted), 
      n_samples = nrow(df_scaled),
      n_features = length(features), 
      Best_ncomp = model$bestTune$ncomp
    )
    
    cat(sprintf("  Finished: %s - %s | CV10_R2: %.3f\n", file, target, tail(summary_list, 1)[[1]]$CV10_R2))
  }
}

# CSV出力
if (length(summary_list) > 0) {
  write.csv(do.call(rbind, summary_list), file.path(run_dir, "all_summary_CV10.csv"), row.names = FALSE)
}
if (length(importance_list) > 0) {
  write.csv(do.call(rbind, importance_list), file.path(run_dir, "all_importance_PLS_VIP.csv"), row.names = FALSE)
}

cat("\n[DONE] PLS Analysis completed.\n")

  Finished: n_base.csv - Jsc | CV10_R2: 0.262
  Finished: n_base.csv - Voc | CV10_R2: 0.705
  Finished: n_base.csv - FF | CV10_R2: 0.151
  Finished: n_base.csv - PCEmax | CV10_R2: 0.244
  Finished: n_base_OH_rebuilt.csv - Jsc | CV10_R2: 0.766
  Finished: n_base_OH_rebuilt.csv - Voc | CV10_R2: 0.839
  Finished: n_base_OH_rebuilt.csv - FF | CV10_R2: 0.498
  Finished: n_base_OH_rebuilt.csv - PCEmax | CV10_R2: 0.727
  Finished: n_base_FP_rebuilt.csv - Jsc | CV10_R2: 0.635
  Finished: n_base_FP_rebuilt.csv - Voc | CV10_R2: 0.794
  Finished: n_base_FP_rebuilt.csv - FF | CV10_R2: 0.400
  Finished: n_base_FP_rebuilt.csv - PCEmax | CV10_R2: 0.619
  Finished: n_all.csv - Jsc | CV10_R2: 0.612
  Finished: n_all.csv - Voc | CV10_R2: 0.640
  Finished: n_all.csv - FF | CV10_R2: 0.276
  Finished: n_all.csv - PCEmax | CV10_R2: 0.526
  Finished: n_all_OH_rebuilt.csv - Jsc | CV10_R2: 0.648
  Finished: n_all_OH_rebuilt.csv - Voc | CV10_R2: 0.740
  Finished: n_all_OH_rebuilt.csv - FF | CV10_R2: 0.321
  Fin